In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from itertools import combinations
from numba import njit, prange

from scipy.special import expit
from scipy.optimize import minimize
from sklearn.metrics import (
    roc_auc_score, f1_score,
    brier_score_loss, log_loss,
    average_precision_score, accuracy_score
)
from scipy.special import comb

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)


In [ ]:
DATA_DIR = Path("../data/email-Enron")
EDGE_CSV = DATA_DIR / "enron_order4_edges.csv"   # columns i,j,k,l

edges = pd.read_csv(EDGE_CSV)
E = np.sort(edges[["i","j","k","l"]].values.astype(int), axis=1)
E = np.unique(E, axis=0)  # dedup

print("Loaded unique 4-edges:", E.shape[0])

# Reindex nodes to 0..n-1
nodes = np.unique(E.reshape(-1))
node2new = {int(u): idx for idx, u in enumerate(nodes)}
E = np.vectorize(node2new.get)(E)

n = len(nodes)
m = E.shape[0]
print("Num nodes:", n)


In [ ]:
def discrete_laplace_noise(eps_per_coordinate, size, rng=None):
    """Sample iid discrete Laplace noise with parameter a = exp(-eps_per_coordinate).
    For an r-uniform hypergraph degree sequence, set eps_per_coordinate = eps / r."""
    rng = np.random.default_rng() if rng is None else rng
    a = np.exp(-eps_per_coordinate)
    mag = rng.geometric(p=1 - a, size=size) - 1
    sign = rng.choice([-1, 1], size=size)
    return mag * sign

def quadru_indices(n):
    """Return all n-choose-4 index quadruples (i,j,k,l) with i<j<k<l, as four arrays."""
    indices = np.array(list(combinations(range(n), 4)), dtype=np.int32)
    return indices[:, 0], indices[:, 1], indices[:, 2], indices[:, 3]

def quadru_pairs(n):
    """Return the (j,k,l) index arrays (with i implicit) for n-choose-4 quadruples."""
    j_all, k_all, l_all = quadru_indices(n)[1:]  # Skip i
    return j_all.astype(np.int32), k_all.astype(np.int32), l_all.astype(np.int32)

def masks_excluding_vertex(n, j_all, k_all, l_all):
    """For each vertex i, return a boolean mask selecting quadruples not containing i."""
    masks = []
    for i in range(n):
        masks.append((j_all != i) & (k_all != i) & (l_all != i))
    return masks

def node_degrees_from_quadruples(quadruples: np.ndarray, n_nodes: int):
    """Compute the degree of each node from an (m, 4) array of hyperedges."""
    deg = np.zeros(n_nodes, dtype=int)
    np.add.at(deg, quadruples.reshape(-1), 1)
    return deg

def make_edge_set(E4: np.ndarray):
    """Build a Python set of hyperedge tuples for O(1) membership tests."""
    return set(map(tuple, E4.tolist()))


In [ ]:
@njit(parallel=True)
def _numba_a_and_grad_r4(beta, n):
    """Numba-accelerated computation of A(beta) = sum_{i<j<k<l} log(1+exp(beta_i+beta_j+beta_k+beta_l))
    and its gradient (expected degrees), parallelized over the outer vertex index."""
    A = 0.0
    grad = np.zeros(n, dtype=np.float64)

    for i in prange(n - 3):
        local_A = 0.0
        local_grad = np.zeros(n, dtype=np.float64)
        for j in range(i + 1, n - 2):
            for k in range(j + 1, n - 1):
                for l in range(k + 1, n):
                    s = beta[i] + beta[j] + beta[k] + beta[l]

                    # logaddexp(0, s), inlined for numba + numerical stability
                    if s > 0:
                        local_A += s + np.log1p(np.exp(-s))
                    else:
                        local_A += np.log1p(np.exp(s))

                    p = 1.0 / (1.0 + np.exp(-s))

                    local_grad[i] += p
                    local_grad[j] += p
                    local_grad[k] += p
                    local_grad[l] += p

        A += local_A
        grad += local_grad

    return A, grad


In [ ]:
def predict_edge_probs(beta, quadruples):
    """Compute link probabilities expit(beta_i+beta_j+beta_k+beta_l) for a batch
    of 4-uniform hyperedges given a fitted parameter vector beta."""
    s = beta[quadruples[:, 0]] + beta[quadruples[:, 1]] + beta[quadruples[:, 2]] + beta[quadruples[:, 3]]
    return expit(s)


In [ ]:
def sample_negative_edges(n, forbidden_set, m, rng):
    """Uniformly sample m vertex quadruples not present in forbidden_set, for
    use as negative (non-edge) examples."""
    neg = []
    seen = set()
    tries = 0
    max_tries = 50 * m + 10_000
    while len(neg) < m and tries < max_tries:
        tries += 1
        quad = rng.choice(n, size=4, replace=False)
        quad.sort()
        t = (int(quad[0]), int(quad[1]), int(quad[2]), int(quad[3]))
        if t in forbidden_set or t in seen:
            continue
        seen.add(t)
        neg.append(t)
    if len(neg) < m:
        raise RuntimeError(
            f"Could not sample enough negative edges: got {len(neg)} of {m}. "
            f"Graph may be too dense for n={n}."
        )
    return np.array(neg, dtype=np.int32)


def ece_score(y_true, y_prob, n_bins=15):
    """Compute the expected calibration error (ECE) of predicted probabilities
    y_prob against binary outcomes y_true, using n_bins equal-width bins."""
    y_true = np.asarray(y_true, dtype=float)
    y_prob = np.asarray(y_prob, dtype=float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    n = y_true.size
    for b in range(n_bins):
        lo, hi = bins[b], bins[b + 1]
        if b == n_bins - 1:
            mask = (y_prob >= lo) & (y_prob <= hi)
        else:
            mask = (y_prob >= lo) & (y_prob < hi)
        if not np.any(mask):
            continue
        frac = mask.mean()
        acc = y_true[mask].mean()
        conf = y_prob[mask].mean()
        ece += frac * abs(acc - conf)
    return float(ece)


In [ ]:
def evaluate_metrics(y_true, p_hat, thresholds=(0.5, 1/3, 0.25), ece_bins=15):
    """Compute ROC-AUC, average precision, Brier score, log-loss, ECE, and
    F1/accuracy at each threshold in `thresholds`."""
    y_true = np.asarray(y_true).astype(int)
    p_hat = np.asarray(p_hat).astype(float)

    out = {}
    out["roc_auc"] = roc_auc_score(y_true, p_hat)
    out["ap"] = average_precision_score(y_true, p_hat)
    out["brier"] = brier_score_loss(y_true, p_hat)

    p_clip = np.clip(p_hat, 1e-15, 1 - 1e-15)
    out["logloss"] = log_loss(y_true, p_clip)

    out["ece"] = ece_score(y_true, p_hat, n_bins=ece_bins)

    for th in thresholds:
        y_pred = (p_hat >= th).astype(int)
        out[f"f1@{th:.2f}"] = f1_score(y_true, y_pred, zero_division=0)
        out[f"acc@{th:.2f}"] = accuracy_score(y_true, y_pred)

    return out


In [ ]:
class HypergraphBeta4:
    """Log-partition function, gradients, and degree-based estimators for the
    4-uniform hypergraph beta-model (A_and_grad is Numba-accelerated; see
    _numba_a_and_grad_r4)."""

    def __init__(self, n):
        self.n = n
        self.r = 4
        self.j_all, self.k_all, self.l_all = quadru_pairs(n)
        self.exclude_masks = masks_excluding_vertex(n, self.j_all, self.k_all, self.l_all)
        self.C = comb(self.n, self.r)  # binom(n,4)

    def A_and_grad(self, beta, chunk_size_pairs=None):
        """Compute A(beta) and its gradient (expected degrees) via the Numba
        kernel; chunk_size_pairs is accepted for interface parity but unused."""
        return _numba_a_and_grad_r4(beta, self.n)

    def ridge_box_fit_from_degrees(self, d_obs, M, lam, beta_init=None,
                                    maxiter=2000, chunk_size_pairs=250_000, verbose=False):
        """Fit beta via projected gradient descent on the ridge-regularized negative
        log-likelihood given observed (possibly noisy) degrees d_obs, then clip to
        [-M, M]. The true M is unknown for real data, so this uses a fixed,
        empirically-stable step size rather than the theoretical worst-case rate.
        """
        n = self.n
        if beta_init is None:
            beta_init = np.zeros(n, dtype=float)

        beta = beta_init.copy()
        eta = 0.005

        if verbose:
            print(f"Starting GD: eta={eta}, iterations={maxiter}")

        for t in range(maxiter):
            _, gA = self.A_and_grad(beta, chunk_size_pairs=chunk_size_pairs)
            grad_obj = (gA - d_obs + lam * beta) / self.C

            beta = beta - eta * grad_obj

            if verbose and t % 500 == 0:
                print(f"Iter {t}: grad_norm={np.linalg.norm(grad_obj)}")

        beta_truncated = np.clip(beta, -M, M)

        return beta_truncated, None

    def box_mle_fit_from_degrees(self, d_obs, M, beta_init=None,
                                maxiter=10000, chunk_size_pairs=250_000, verbose=False):
        """Box-constrained MLE: special case of ridge_box_fit_from_degrees with lam=0."""
        return self.ridge_box_fit_from_degrees(
            d_obs=d_obs, M=M, lam=0.0, beta_init=beta_init,
            maxiter=maxiter, chunk_size_pairs=chunk_size_pairs, verbose=verbose
        )

    def grad_ell(self, beta, d_true, chunk_size_pairs=250_000):
        """Gradient of the (unregularized) negative log-likelihood at beta,
        given the observed degree sequence d_true."""
        _, gA = self.A_and_grad(beta, chunk_size_pairs=chunk_size_pairs)
        return (gA - d_true) / self.C


def central_dp_gd(model, d_true, n, r, M, eps, delta,
                   chunk_size_pairs=250_000, seed=None, beta0=None, T_cap=1000, verbose=False):
    """Differentially private gradient descent estimator for beta under central
    (eps, delta)-edge differential privacy: adds Gaussian noise to the gradient
    at each step, with iteration count and noise scale calibrated so the final
    iterate is (eps, delta)-DP (step size fixed at 0.005, as above)."""
    rng = np.random.default_rng(seed)
    beta = np.zeros(n, dtype=float) if beta0 is None else beta0.astype(float).copy()

    eta = 0.005
    T_init = int(np.ceil(32.0 * (r - 1) * np.exp(4.0 * r * M) * ((r - 1) * np.log(n) + 2.0 * np.log(M))))
    T = min(T_init, T_cap)

    sigma2 = 4.0 * r * T * (n ** (-2.0 * r)) * (eps ** (-2.0)) * np.log(1.0 / delta)
    sigma = float(np.sqrt(sigma2))

    for t in range(T):
        if verbose and ((t + 1) % 500 == 0 or (t + 1) == T):
            print(f"Iter {t+1}/{T} eps={eps}", flush=True)
        g = model.grad_ell(beta, d_true, chunk_size_pairs=chunk_size_pairs)
        z = rng.normal(0.0, sigma, size=n)
        beta = beta - eta * (g + z)

    beta = np.clip(beta, -M, M)

    return beta, dict(eta=float(eta), T=int(T), sigma=float(sigma))


In [ ]:
def local_dp_fit_from_degrees(model, d_true, M, lam, eps_loc, seed=0,
                              maxiter=10000, chunk_size_pairs=250_000, verbose=False):
    """Local DP: privatize the degree vector d_true with discrete Laplace noise,
    then fit the ridge/box estimator using the privatized degrees."""
    rng = np.random.default_rng(seed)
    z = discrete_laplace_noise(eps_per_coordinate=eps_loc, size=len(d_true), rng=rng)
    d_priv = d_true.astype(float) + z.astype(float)
    d_priv = np.maximum(d_priv, 0.0)

    beta_loc, _ = model.ridge_box_fit_from_degrees(
        d_obs=d_priv, M=M, lam=lam, beta_init=None,
        maxiter=maxiter, chunk_size_pairs=chunk_size_pairs, verbose=verbose
    )
    return beta_loc, {"eps_loc": float(eps_loc)}


In [ ]:
# Hold out 10% of observed hyperedges as test positives; the rest form the
# training graph. Test negatives are an equal number of non-edges.
rng = np.random.default_rng(123)

perm = rng.permutation(m)
m_test_pos = int(np.ceil(0.10 * m))
test_pos_idx = perm[:m_test_pos]
train_pos_idx = perm[m_test_pos:]

E_train = E[train_pos_idx]   # observed edges for fitting
E_test_pos = E[test_pos_idx] # held-out positives

# Forbid sampling any observed positive (including held-out) as a negative
obs_set = make_edge_set(E)
forbidden_test = set(obs_set)
E_test_neg = sample_negative_edges(n, forbidden_test, E_test_pos.shape[0], rng)

print("Train edges:", E_train.shape[0])
print("Test positives:", E_test_pos.shape[0])
print("Test negatives:", E_test_neg.shape[0])


In [ ]:
d_train = node_degrees_from_quadruples(E_train, n).astype(float)
print("Mean train degree:", d_train.mean(), "Max:", d_train.max())


In [ ]:
r = 4
model = HypergraphBeta4(n)

# The true M is unknown for real data; use the conservative choice M = 2*sqrt(log n).
M = np.sqrt(np.log(n)) * 2.0
maxiter_mle = 10000
chunk_size_pairs = 250_000

delta = n ** (-2.0)
lam_ridge = 0.0002 * (n ** ((r - 1) / 2))   # for r=4, this is 0.0002*n^1.5

print("delta =", delta)
print("lam_ridge =", lam_ridge)

# Build the unified test set once
E_test = np.vstack([E_test_pos, E_test_neg])
y_test = np.concatenate([
    np.ones(E_test_pos.shape[0], dtype=int),
    np.zeros(E_test_neg.shape[0], dtype=int)
])

# ---- Method 1: Non-private MLE (box-constrained) ----
beta_mle, _ = model.box_mle_fit_from_degrees(
    d_obs=d_train, M=M, beta_init=None,
    maxiter=maxiter_mle, chunk_size_pairs=chunk_size_pairs, verbose=True
)

# ---- Method 2: Non-private ridge MLE (box-constrained) ----
beta_mle_lam, _ = model.ridge_box_fit_from_degrees(
    d_obs=d_train, M=M, lam=lam_ridge, beta_init=None,
    maxiter=maxiter_mle, chunk_size_pairs=chunk_size_pairs, verbose=True
)

# Evaluate non-private baselines once
p_mle = predict_edge_probs(beta_mle, E_test)
p_mle_lam = predict_edge_probs(beta_mle_lam, E_test)

rows = []
rows.append({"method": "MLE", "eps": np.nan, "delta": np.nan, "lam": 0.0, "M": M, **evaluate_metrics(y_test, p_mle)})
rows.append({"method": "MLE_lam", "eps": np.nan, "delta": np.nan, "lam": lam_ridge, "M": M, **evaluate_metrics(y_test, p_mle_lam)})

# ---- Methods 3 & 4: DP methods over an eps grid ----
eps_grid = [0.001, 0.01, 0.1, 1.0]

for eps in eps_grid:
    # Central DP-GD
    beta_cen, info_cen = central_dp_gd(
        model=model, d_true=d_train, n=n, r=r, M=M, eps=eps, delta=delta,
        chunk_size_pairs=chunk_size_pairs, seed=999, T_cap=10000, verbose=True
    )
    p_cen = predict_edge_probs(beta_cen, E_test)
    rows.append({
        "method": "CEN_DP",
        "eps": eps,
        "delta": delta,
        "lam": 0.0,
        "M": M,
        "T": info_cen.get("T", np.nan),
        "sigma": info_cen.get("sigma", np.nan),
        **evaluate_metrics(y_test, p_cen)
    })

    # Local DP (privatize degrees then fit ridge/box)
    beta_loc, info_loc = local_dp_fit_from_degrees(
        model=model, d_true=d_train, M=M, lam=lam_ridge, eps_loc=eps,
        seed=2026, maxiter=maxiter_mle, chunk_size_pairs=chunk_size_pairs, verbose=True
    )
    p_loc = predict_edge_probs(beta_loc, E_test)
    rows.append({
        "method": "LOC_DP",
        "eps": eps,
        "delta": np.nan,     # local DP does not use delta in this degree-noise construction
        "lam": lam_ridge,
        "M": M,
        **evaluate_metrics(y_test, p_loc)
    })

res = pd.DataFrame(rows)
res


In [ ]:
res.to_csv("link_prediction_results_order4.csv", index=False)
